In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [2]:
# https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
model_path = 'hand_landmarker.task'
DUMMY_VALUE = 0

In [3]:
# Visualization And Plotting Utils
def fit_world_to_image(world_matrix, norm_matrix, w, h):
    """Least-squares affine map  [wx, wy, wz, 1] -> [px, py]  over all 21 landmarks."""
    pixels = norm_matrix * [w, h]
    W = np.hstack([world_matrix, np.ones((len(world_matrix), 1))])
    A_T, *_ = np.linalg.lstsq(W, pixels, rcond=None)
    return A_T.T   # (2, 4)

def draw_vector_on_frame(frame_bgr, hand_norm, hand_world, vector,
                         anchor_indices=(0, 9), anchor_weights=(0.5, 0.5),
                         scale=10.0, color=(0, 255, 0)):
    h, w, _ = frame_bgr.shape
    norm  = np.array([[lm.x, lm.y]       for lm in hand_norm])
    world = np.array([[lm.x, lm.y, lm.z] for lm in hand_world])

    A = fit_world_to_image(world, norm, w, h)
    A_linear = A[:, :3]

    a_idx = np.asarray(anchor_indices)
    a_wts = np.asarray(anchor_weights, dtype=float).reshape(-1, 1)
    tail = ((a_wts * norm[a_idx]).sum(axis=0)) * [w, h]
    tip  = tail + (A_linear @ np.asarray(vector)) * scale
    _2dvec = A_linear @ np.asarray(vector)
    p1 = (int(tail[0]), int(tail[1]))
    p2 = (int(tip[0]),  int(tip[1]))
    cv2.arrowedLine(frame_bgr, p1, p2, color, 3, tipLength=0.3)
    return _2dvec

In [4]:
# Feature Calculations
HAND_BONES = [(0, 5), (5, 17)]
vectors = []
def bone_vectors(world_matrix, bones=HAND_BONES):
    vecs = np.array([world_matrix[e] - world_matrix[s] for (s, e) in bones])
    return vecs

def compute_vector(world):
    vecs = bone_vectors(world)
    result = np.cross(vecs[0], vecs[1])
    return result

In [10]:
# Using Colin's parameters
video_options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO, 
    min_hand_detection_confidence=0.7,
    min_tracking_confidence=0.3,
    num_hands=2,)
video_detector = vision.HandLandmarker.create_from_options(video_options)

I0000 00:00:1782062616.189475       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M5


In [11]:
# output a video
cap = cv2.VideoCapture("../hmm-testing/picklist_videos/picklist_171.mp4")          
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter("picklist_273_palm_up.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"),  
                         fps, (width, height))
vectors = []
angles = []
depth_per_frame = []   # one entry per frame, NaN when no usable hand
frame_idx = 0
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    timestamp_ms = int(1000 * frame_idx / fps)
    result = video_detector.detect_for_video(mp_image, timestamp_ms)

    if result.hand_landmarks:
        n = len(result.hand_landmarks)
        if n == 1:
            label   = result.handedness[0][0].category_name
            wrist_x = result.hand_landmarks[0][0].x
            idx = None if (label == "Left" and wrist_x <= 0.5) else 0
        else:
            idx = max(range(n), key=lambda i: result.hand_landmarks[i][0].x)

        if idx is None:
            vectors.append(np.array([DUMMY_VALUE, DUMMY_VALUE, DUMMY_VALUE]))
            depth_per_frame.append(np.nan)          # no hand -> no value
        else:
            hand_norm  = result.hand_landmarks[idx]
            hand_world = result.hand_world_landmarks[idx]
            label      = result.handedness[idx][0].category_name

            world = np.array([[lm.x, lm.y, lm.z] for lm in hand_world])
            if label == "Left":
                world[:, 0] *= -1
            vec = compute_vector(world)
            vectors.append(vec)
            draw_vector_on_frame(frame_bgr, hand_norm, hand_world, vec)

            vec_norm = vec / np.linalg.norm(vec)
            out_of_screen = float(vec_norm[2])
            angles.append(out_of_screen)
            depth_per_frame.append(out_of_screen)   # frame-aligned copy
    else:
        vectors.append(np.array([DUMMY_VALUE, DUMMY_VALUE, DUMMY_VALUE]))
        depth_per_frame.append(np.nan)

    writer.write(frame_bgr)
    frame_idx += 1

cap.release()
writer.release()

In [12]:
# output a depth
import csv

with open("hand_depth.csv", "w", newline="") as f:
    w = csv.writer(f)
    for i, val in enumerate(depth_per_frame):
        t = i / fps                      # seconds, regular interval
        if np.isnan(val):
            w.writerow([f"{t:.3f}", ""]) # leave value blank on no-hand frames
        else:
            w.writerow([f"{t:.3f}", f"{val:.5f}"])